# Task 3: Transfer Learning, Layer Freezing & Fine-Tuning
____________________________________________________________________________________________________

We use a **ResNet-18 pre-trained on ImageNet** (1.2M images, 1000 classes) and adapt it to CIFAR-10 in 3 ways, then compare with our best CNN trained from scratch (Task 2):

| setup | what is trained | learning rate |
|---|---|---|
| **scratch** | our best CustomCNN, all layers, random init | 1e-3 |
| **frozen** (feature extraction) | only the new classification head, backbone frozen (`requires_grad = False`) | 1e-3 |
| **finetune_layer4** | last residual block (`layer4`) + head | 1e-4 (reduced) |
| **finetune_all** | the whole network | 1e-4 (reduced) |

The fine-tuning uses a **reduced learning rate** so the pre-trained filters are only slightly adapted and not destroyed by big updates.

Each setup is trained with **10% and 100%** of the training images to measure **data efficiency**. We also compare **convergence speed** (epochs to reach a given val accuracy) and the **final test metrics**.

All runs use the same recipe: AdamW, cosine scheduler, basic augmentation (crop + flip), 10 epochs.

## 0. Google Colab setup

Only runs on Google Colab (skipped automatically when running locally).

**Before the first run**, the repository must be cloned into Google Drive (one time only, see the README):
```python
!git clone -b course-project https://github.com/MohammedZaiter/ai.git /content/drive/MyDrive/ai
```

This cell then:
1. mounts Google Drive, so checkpoints and results are saved in `My Drive/ai` and are not lost when the session ends
2. moves into the `Notebooks/` folder so that `import utils` and the relative paths work

On Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = "/content/drive/MyDrive/ai"  # folder of the project on Google Drive
    os.chdir(PROJECT_DIR + "/Notebooks")
    sys.path.insert(0, os.getcwd())  # so that "import utils" finds utils.py
    print("Working directory:", os.getcwd())

In [ ]:
import os

import torch
import torch.nn as nn
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

import utils
from utils import classes

## 1. Settings

In [ ]:
TRAIN = True

num_epochs = 10
batch_size = 128
SEED = 42

# ResNet-18 was trained on 224x224 images. CIFAR images (32x32) are upscaled to 128x128:
# big enough for the pre-trained filters, and much faster than 224x224
IMG_SIZE = 128

train_fractions = [0.1, 1.0]

# best scratch architecture from Task 2 (update it if you changed t2_best)
SCRATCH_MODEL_ARGS = {"conv_channels": [64, 128, 256], "use_bn": True, "dropout": 0.3}

setups = {
    "scratch":         {"lr": 1e-3},
    "frozen":          {"lr": 1e-3},
    "finetune_layer4": {"lr": 1e-4},
    "finetune_all":    {"lr": 1e-4},
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

criterion = nn.CrossEntropyLoss()

## 2. Freezing: which parameters are trained?

`param.requires_grad = False` means no gradient is computed for this parameter, so the optimizer never changes it. The new head (`model.fc`) is created **after** the freezing loop, so it is always trainable.

In [ ]:
for mode in ["frozen", "finetune_layer4", "finetune_all"]:
    model = utils.get_resnet18(mode)
    total = utils.count_parameters(model)
    trainable = utils.count_parameters(model, trainable_only=True)
    print(f"{mode:16s} trainable: {trainable:>10,} / {total:,} ({100 * trainable / total:.1f} %)")

# names of the trainable layers in the "finetune_layer4" setup
model = utils.get_resnet18("finetune_layer4")
trainable_layers = set(name.split(".")[0] for name, param in model.named_parameters() if param.requires_grad)
print("\nfinetune_layer4 -> trainable layers:", trainable_layers)

In [ ]:
# new classification head (replaces the 1000-class ImageNet layer)
print(model.fc)

**Input images for ResNet:** upscaled to 128x128 and normalized with the **ImageNet** mean/std, because the pre-trained filters expect the same input distribution as during their training.

In [ ]:
sample_loader, _, _ = utils.get_dataloaders(batch_size=8, img_size=IMG_SIZE, imagenet_norm=True)
images, labels = next(iter(sample_loader))
print("batch shape:", images.shape)
utils.imshow(torchvision.utils.make_grid(images, nrow=8), imagenet_norm=True)
print(' '.join('%5s' % classes[labels[j]] for j in range(8)))

## 3. Function to run one experiment

In [ ]:
def run_experiment(setup, train_fraction):
    name = f"t3_{setup}_{int(train_fraction * 100)}pct"
    checkpoint_path = f"{utils.CHECKPOINT_DIR}/{name}.pth"
    history_path = f"{utils.RESULTS_DIR}/{name}.json"
    lr = setups[setup]["lr"]

    # scratch CNN works on the original 32x32 images, ResNet on 128x128 with ImageNet normalization
    if setup == "scratch":
        img_size, imagenet_norm = 32, False
    else:
        img_size, imagenet_norm = IMG_SIZE, True

    utils.set_seed(SEED)
    train_loader, val_loader, test_loader = utils.get_dataloaders(batch_size=batch_size, augmentation="basic",
                                                                  img_size=img_size, imagenet_norm=imagenet_norm,
                                                                  train_fraction=train_fraction)

    if TRAIN and not os.path.exists(history_path):
        print(f"\n===== Training {name} ({len(train_loader.dataset)} training images) =====")
        if setup == "scratch":
            model = utils.CustomCNN(**SCRATCH_MODEL_ARGS).to(device)
            checkpoint_info = {"model_name": "CustomCNN", "model_args": SCRATCH_MODEL_ARGS}
        else:
            model = utils.get_resnet18(setup).to(device)
            checkpoint_info = {"model_name": "resnet18", "model_args": {"mode": setup},
                               "img_size": img_size, "imagenet_norm": imagenet_norm}

        # only the trainable parameters are given to the optimizer
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = utils.get_optimizer("adamw", trainable_params, lr)
        scheduler = utils.get_scheduler("cosine", optimizer, num_epochs)

        history = utils.train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device,
                                    checkpoint_path, scheduler=scheduler, checkpoint_info=checkpoint_info)
        utils.save_history(history, history_path)
    else:
        print(f"Loading {name}")
        history = utils.load_history(history_path)

    # best checkpoint -> test set
    model, _ = utils.load_checkpoint(checkpoint_path, device)
    test_loss, test_acc, test_class_acc, test_preds, test_labels = utils.evaluate(model, test_loader, criterion, device)

    result = {
        "name": name,
        "setup": setup,
        "train fraction": train_fraction,
        "total params": utils.count_parameters(model),
        "trainable params": utils.count_parameters(model, trainable_only=True),  # same freezing as in training
        "peak val acc": max(history["val_acc"]),
        "epochs to 70% val acc": utils.epochs_to_reach(history, 0.70),
        "epochs to 80% val acc": utils.epochs_to_reach(history, 0.80),
        "test acc": test_acc,
        "test macro F1": f1_score(test_labels, test_preds, average="macro"),
        "time/epoch (s)": np.mean(history["epoch_time"]),
        "latency (ms)": utils.measure_latency(model, device, img_size),
    }
    return result, history, (test_preds, test_labels, test_class_acc)

## 4. Run all experiments

4 setups x 2 train fractions = 8 runs. The ResNet runs are slower (bigger model and 128x128 images).

In [ ]:
results = []
histories = {}
test_outputs = {}

for train_fraction in train_fractions:
    for setup in setups:
        result, history, test_output = run_experiment(setup, train_fraction)
        results.append(result)
        histories[result["name"]] = history
        test_outputs[result["name"]] = test_output

## 5. Comparison table

In [ ]:
results_df = pd.DataFrame(results).set_index("name")
results_df.to_csv(f"{utils.RESULTS_DIR}/task3_comparison.csv")
results_df.round(4)

## 6. Convergence speed

Validation accuracy per epoch for each setup (100% of the training data). Pre-trained models start from useful features, so they should reach a high accuracy after very few epochs.

In [ ]:
for train_fraction in train_fractions:
    pct = int(train_fraction * 100)
    utils.plot_compare({setup: histories[f"t3_{setup}_{pct}pct"] for setup in setups},
                       title=f"{pct}% of training data", save_name=f"task3_convergence_{pct}pct.png")

## 7. Data efficiency

Test accuracy with 10% (4,500 images) vs 100% (45,000 images) of the training set. A small drop between 100% and 10% means the method needs less data.

In [ ]:
data_eff = results_df.pivot(index="setup", columns="train fraction", values="test acc").loc[list(setups)]
data_eff.columns = [f"{int(c * 100)}% data" for c in data_eff.columns]
print(data_eff.round(4))

data_eff.plot.bar(figsize=(8, 4), rot=0)
plt.ylabel("Test accuracy")
plt.title("Data efficiency: test accuracy vs amount of training data")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
utils.save_figure("task3_data_efficiency.png")
plt.show()

## 8. Learning curves and confusion matrix of each setup (100% data)

In [ ]:
for setup in setups:
    name = f"t3_{setup}_100pct"
    utils.plot_history(histories[name], title=name, save_name=f"{name}_curves.png")

In [ ]:
for setup in ["scratch", "frozen", "finetune_all"]:
    name = f"t3_{setup}_100pct"
    test_preds, test_labels, test_class_acc = test_outputs[name]
    utils.plot_confusion_matrix(test_labels, test_preds, title=f"{name} - normalized confusion matrix (test)",
                                save_name=f"{name}_confusion_matrix.png")

In [ ]:
# per-class test accuracy of each setup (100% data)
class_acc_df = pd.DataFrame({setup: test_outputs[f"t3_{setup}_100pct"][2] for setup in setups}, index=classes)
class_acc_df.plot.bar(figsize=(12, 4), rot=0)
plt.ylabel("Test accuracy")
plt.title("Per-class test accuracy (100% data)")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
utils.save_figure("task3_class_acc.png")
plt.show()

## Observations

*(to fill after training, with numbers from the tables)*

- **Final test metrics:** scratch vs frozen vs fine-tuned
- **Convergence speed:** how many epochs does each setup need to reach 70% / 80%?
- **Data efficiency:** which setup loses the least accuracy with only 10% of the data? Why?
- **Frozen vs fine-tuned:** ImageNet features are generic (edges, textures, shapes); fine-tuning adapts the upper layers to CIFAR-10 objects and small upscaled images
- **Cost:** trainable parameters, time per epoch and latency of each setup